In this notebook, you should implement a first version of a working machine learning model to predict the age of an Abalone.

A few guidelines:
- The model does not have to be complex. A simple linear regression model is enough.
- You should use MLflow to track your experiments. You can use the MLflow UI to compare your experiments.
- Do not push any MLflow data to the repository. Only the code to run the experiments is interesting and should be pushed.

In [ ]:
# %% Imports and load
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
df = pd.read_csv(project_root / "data" / "abalone.csv")

# %% Target: age = rings + 1.5, then split features / target
df["Age"] = df["Rings"] + 1.5
X = df.drop(columns=["Rings", "Age"])
y = df["Age"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# %% Model pipeline: one-hot encode categoricals, scale numerics, linear regression
preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ]
)
pipe = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])

# %% MLflow setup (local filesystem, no tracking server)
mlflow.set_experiment("abalone-age-prediction")

# %% Train and track a run
with mlflow.start_run(run_name="linear-regression-baseline"):
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    # Metrics
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    # Params
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("features", list(X.columns))

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # Model (with input/output signature)
    signature = infer_signature(X_train, pipe.predict(X_train))
    mlflow.sklearn.log_model(
        pipe,
        name="model",
        signature=signature,
        skops_trusted_types=["sklearn.compose._column_transformer._RemainderColsList"],
    )

    print(f"RMSE: {rmse:.3f} | MAE: {mae:.3f} | R2: {r2:.3f}")

# %% (Optional) Try another run to compare in the UI, e.g. log-transformed target
with mlflow.start_run(run_name="linear-regression-log-target"):
    pipe.fit(X_train, np.log(y_train))
    preds = np.exp(pipe.predict(X_test))

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("target_transform", "log")
    mlflow.log_metric("rmse", np.sqrt(mean_squared_error(y_test, preds)))
    mlflow.log_metric("mae", mean_absolute_error(y_test, preds))
    mlflow.log_metric("r2", r2_score(y_test, preds))

# %% Launch the UI from a terminal (not the notebook), in the folder containing "mlruns":
#   mlflow ui
# then open http://127.0.0.1:5000 and compare the runs.

RMSE: 2.212 | MAE: 1.593 | R2: 0.548
